¡Claro que sí! Es un proyecto excelente para automatizar la recolección de datos en el aula. 

Para que una aplicación hecha en Python funcione en un celular (Android), necesitamos usar un framework llamado **Kivy**. Kivy permite crear interfaces gráficas que luego se pueden empaquetar como un archivo `.apk` (el instalador de Android) que podrías enviar por WhatsApp.

Aquí tienes el código base y los pasos para lograrlo.

---

## 1. El Código Python (main.py)
Copia este código en un archivo llamado `main.py` dentro de **VS Code**. Este script genera la interfaz, maneja las preguntas de la 41 a la 78 y guarda el archivo CSV.



In [2]:
from kivy.app import App
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.label import Label
from kivy.uix.togglebutton import ToggleButton
from kivy.uix.button import Button
from kivy.uix.scrollview import ScrollView
from kivy.uix.gridlayout import GridLayout
import csv
import os

class ExamenApp(App):
    def build(self):
        # Diccionario para almacenar las respuestas {numero_pregunta: opcion}
        self.respuestas = {i: "" for i in range(41, 79)}
        
        layout_principal = BoxLayout(orientation='vertical', padding=10, spacing=10)
        
        # Título
        layout_principal.add_widget(Label(
            text="Examen: Preguntas 41 a 78", 
            size_hint_y=None, 
            height=50, 
            font_size='20sp',
            bold=True
        ))
        
        # Área de desplazamiento
        scroll = ScrollView()
        lista_preguntas = GridLayout(cols=1, spacing=25, size_hint_y=None, padding=[0, 10, 0, 10])
        lista_preguntas.bind(minimum_height=lista_preguntas.setter('height'))
        
        for i in range(41, 79):
            bloque = BoxLayout(orientation='vertical', size_hint_y=None, height=100)
            bloque.add_widget(Label(text=f"Pregunta {i}", halign="left", size_hint_y=None, height=30))
            
            opciones_layout = BoxLayout(orientation='horizontal', spacing=10)
            for opcion in ['A', 'B', 'C', 'D']:
                # Usamos ToggleButton para que el grupo funcione correctamente
                btn = ToggleButton(
                    text=opcion, 
                    group=str(i),  # Esto agrupa las opciones por pregunta
                    background_normal='', 
                    background_color=(0.3, 0.3, 0.3, 1),
                    background_down='', # Color cuando está seleccionado
                    color=(1, 1, 1, 1)
                )
                # Cambiamos el color visual al presionar y guardamos la respuesta
                btn.bind(on_release=lambda btn, q=i, opt=opcion: self.registrar_respuesta(q, opt, btn))
                opciones_layout.add_widget(btn)
            
            bloque.add_widget(opciones_layout)
            lista_preguntas.add_widget(bloque)
            
        scroll.add_widget(lista_preguntas)
        layout_principal.add_widget(scroll)
        
        # Botón de Guardar
        self.btn_finalizar = Button(
            text="Finalizar y Generar CSV", 
            size_hint_y=None, 
            height=70, 
            background_color=(0.1, 0.7, 0.3, 1),
            bold=True
        )
        self.btn_finalizar.bind(on_release=self.generar_csv)
        layout_principal.add_widget(self.btn_finalizar)
        
        return layout_principal

    def registrar_respuesta(self, pregunta, opcion, boton):
        if boton.state == 'down':
            self.respuestas[pregunta] = opcion
        else:
            self.respuestas[pregunta] = "" # Si lo desmarca

    def generar_csv(self, instance):
        # Nombre del archivo basado en el almacenamiento de la app
        nombre_archivo = "respuestas_estudiante.csv"
        try:
            with open(nombre_archivo, mode='w', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(["Pregunta", "Respuesta Seleccionada"])
                for i in range(41, 79):
                    writer.writerow([i, self.respuestas[i]])
            
            self.btn_finalizar.text = "¡CSV Creado con éxito!"
            self.btn_finalizar.background_color = (0.2, 0.2, 0.8, 1)
            print(f"Archivo guardado exitosamente.")
        except Exception as e:
            self.btn_finalizar.text = "Error al guardar"
            print(f"Error: {e}")

if __name__ == '__main__':
    ExamenApp().run()

OSError: source code not available


## 2. ¿Cómo convertirlo en una App para WhatsApp?

Python no corre de forma nativa en Android como un `.exe` en Windows. Para convertir este código en un archivo `.apk` (que es el que enviarás por WhatsApp), debes seguir estos pasos:

### Paso A: Probar en VS Code
1. Instala Kivy en tu terminal: `pip install kivy`.
2. Ejecuta el código para asegurarte de que la interfaz se ve como deseas.

### Paso B: Empaquetar para Android (Buildozer)
Esta es la parte técnica. Android requiere un proceso de compilación. La herramienta más común es **Buildozer**.
1. **Importante:** Buildozer solo funciona en **Linux** (o en Windows usando WSL2 - Subsistema de Linux).
2. En la terminal de Linux/WSL2, instalas Buildozer.
3. Ejecutas `buildozer init`, lo que creará un archivo `buildozer.spec`.
4. Editas ese archivo para ponerle el nombre a tu App (ej. "ExamenEstudiantes").
5. Ejecutas `buildozer -v android debug`.

---



## 3. Consideraciones importantes para tus alumnos

* **Ubicación del archivo:** En Android, el archivo `.csv` se guardará en la carpeta interna de la aplicación. Para que los alumnos puedan enviártelo de vuelta, tendrías que agregar una función de "Compartir archivo" usando una librería llamada `Plyer`.
* **Orígenes desconocidos:** Cuando envíes el `.apk` por WhatsApp, el celular de tus alumnos les advertirá que es una "aplicación de origen desconocido". Ellos deberán autorizar la instalación en los ajustes de seguridad de su teléfono.

> **Sugerencia de "Plan B":** Si el proceso de compilar a APK te resulta muy complejo, podrías usar **Google Forms**. Es mucho más sencillo de enviar por WhatsApp, no requiere instalación y te genera el CSV (en Google Sheets) automáticamente sin que los estudiantes tengan que enviarte archivos individuales. 

¿Te gustaría que te ayude a ajustar el código para que el archivo CSV se guarde en una carpeta pública del celular y sea más fácil de encontrar?